In [7]:
import soccerdata as sd
import pandas as pd

# 1. Descargamos el fixture de ambas temporadas desde Understat
understat = sd.Understat(leagues="ENG-Premier League", seasons=["1718", "1920"])
schedule = understat.read_schedule().reset_index()

# Verificamos qué columnas trajo efectivamente
print("Columnas disponibles:")
print(schedule.columns.tolist())

# 2. Filtramos los partidos donde jugó Liverpool
lfc_raw = schedule[(schedule["home_team"] == "Liverpool") | (schedule["away_team"] == "Liverpool")].copy()

# 3. Normalizamos goles y xG desde la perspectiva del Liverpool
records = []
for _, row in lfc_raw.iterrows():
    is_home = row["home_team"] == "Liverpool"
    
    records.append({
        "season": row["season"],
        "date": row["date"],
        "opponent": row["away_team"] if is_home else row["home_team"],
        "is_home": is_home,
        "goals_for": row["home_goals"] if is_home else row["away_goals"],
        "goals_against": row["away_goals"] if is_home else row["home_goals"],
        "xg_for": row["home_xg"] if is_home else row["away_xg"],
        "xg_against": row["away_xg"] if is_home else row["home_xg"]
    })

df = pd.DataFrame(records)

print("\nPartidos cargados por temporada:")
print(df["season"].value_counts())
df.head()

[10/18/26 17:55:45] INFO     Saving cached data to C:\Users\Win10\soccerdata\data\Understat          ]8;id=3143396;file://C:\Users\Win10\AppData\Roaming\Python\Python314\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=3143397;file://C:\Users\Win10\AppData\Roaming\Python\Python314\site-packages\soccerdata\_common.py#250\250]8;;\

Columnas disponibles:
['league', 'season', 'game', 'league_id', 'season_id', 'game_id', 'date', 'home_team_id', 'away_team_id', 'home_team', 'away_team', 'away_team_code', 'home_team_code', 'home_goals', 'away_goals', 'home_xg', 'away_xg', 'is_result', 'has_data', 'url']

Partidos cargados por temporada:
season
1718    38
1920    38
Name: count, dtype: int64


,season,date,opponent,is_home,goals_for,goals_against,xg_for,xg_against
0,1718,2017-08-12 12:30:00,Watford,False,3,3,2.615490,2.176470
1,1718,2017-08-19 15:00:00,Crystal Palace,True,1,0,2.849780,0.727364
2,1718,2017-08-27 16:00:00,Arsenal,True,4,0,3.459980,0.484816
3,1718,2017-09-09 12:30:00,Manchester City,False,0,5,0.665186,3.145890
4,1718,2017-09-16 15:00:00,Burnley,True,1,1,2.187270,0.334402


In [8]:
# Agrupamos por temporada y sacamos el promedio por partido
resumen_promedios = df.groupby("season")[["goals_for", "xg_for", "goals_against", "xg_against"]].mean()

# Le damos formato lindo a 2 decimales
resumen_promedios.round(2)

,goals_for,xg_for,goals_against,xg_against
season,,,,
1718,2.21,2.04,1.00,0.94
1920,2.24,1.98,0.87,1.04


In [9]:
# Sumamos el total de toda la temporada (los 38 partidos)
totales = df.groupby("season")[["goals_for", "xg_for", "goals_against", "xg_against"]].sum()

# Creamos dos columnas nuevas para ver la diferencia:
# Si 'diff_ataque' es positivo -> metieron MÁS goles de los que marcaba la probabilidad de sus tiros
totales["diff_ataque"] = totales["goals_for"] - totales["xg_for"]

# Si 'diff_defensa' es negativo -> recibieron MENOS goles de los que marcaba el peligro rival (arquerazo o suerte)
totales["diff_defensa"] = totales["goals_against"] - totales["xg_against"]

totales.round(1)

,goals_for,xg_for,goals_against,xg_against,diff_ataque,diff_defensa
season,,,,,,
1718,84,77.5,38,35.8,6.5,2.2
1920,85,75.2,33,39.6,9.8,-6.6


In [10]:
# 1. Creamos una columna temporal con los goles totales de cada partido
df["total_goals"] = df["goals_for"] + df["goals_against"]

# 2. Varianza y desviación estándar de goles totales por partido
dispersion = df.groupby("season")["total_goals"].agg(["mean", "std"])
print("Promedio y Desviación estándar de goles por partido:")
print(dispersion.round(2))

# 3. ¿Cuántos partidos tuvieron 4 goles o más? (partidos de 3-1, 4-2, 2-2, etc.)
partidos_locos = df[df["total_goals"] >= 4].groupby("season")["date"].count()
print("\nCantidad de partidos con 4 o más goles (de 38 jugados):")
print(partidos_locos)

Promedio y Desviación estándar de goles por partido:
        mean   std
season            
1718    3.21  1.86
1920    3.11  1.56

Cantidad de partidos con 4 o más goles (de 38 jugados):
season
1718    16
1920    13
Name: date, dtype: int64
